In [35]:
# =========================================================
# 1) Imports
# =========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
import pymc as pm
import arviz as az


In [37]:
# =========================================================
# 2) Função de preparação
# =========================================================
def preparar_dados(df: pd.DataFrame) -> pd.DataFrame:
    """Cria variáveis derivadas e prepara dados para modelagem."""
    df = df.copy()
    df.columns = [c.lower() for c in df.columns]

    # Total de docentes
    df["qt_doc_total_calc"] = (
        df["qt_doc_ex_grad"].fillna(0)
        + df["qt_doc_ex_esp"].fillna(0)
        + df["qt_doc_ex_mest"].fillna(0)
        + df["qt_doc_ex_dout"].fillna(0)
    )

    # Proporção de docentes avançados
    df["prop_doc_avancado"] = (
        (df["qt_doc_ex_mest"].fillna(0) + df["qt_doc_ex_dout"].fillna(0))
        / df["qt_doc_total_calc"].replace(0, np.nan)
    )

    # Proporções de ingressantes
    df["prop_ing_pp"] = (df["qt_ing_preta"] + df["qt_ing_parda"]) / df["qt_ing"].replace(0, np.nan)
    df["prop_ing_financiados"] = (
        df["qt_ing_fies"] + df["qt_ing_prounii"] + df["qt_ing_prounip"]
    ) / df["qt_ing"].replace(0, np.nan)

    # Tratar NaN e limitar
    for col in ["prop_doc_avancado", "prop_ing_pp", "prop_ing_financiados"]:
        df[col] = df[col].fillna(0).clip(0, 1)

    # Remover linhas inválidas
    df_model = df[(df["qt_ing"].notnull()) & (df["qt_ing"] >= 0) & (df["qt_mat"] > 0)].copy()

    # Criar offset
    df_model["offset_log_qtmat"] = np.log(df_model["qt_mat"])

    return df_model


In [38]:
# =========================================================
# 3) Carregar e preparar dados
# =========================================================
df = pd.read_csv("assets/ride_df_integrado_2023.csv")
df = df[df["nu_ano_censo"] == 2023].copy()
df_model = preparar_dados(df)

print("Shape:", df_model.shape)
df_model.head()


Shape: (5427, 109)


,nu_ano_censo,co_municipio_ies,nome_municipio,municipio_capital,longitude,latitude,sigla_uf,nome_uf,nome_regiao,in_capital_ies,...,taxa_conclusao,taxa_ingresso,perc_feminino,perc_doutores,relacao_candidato_vaga,qt_doc_total_calc,prop_doc_avancado,prop_ing_pp,prop_ing_financiados,offset_log_qtmat
0,2023,5200258,Águas Lindas de Goiás,Não,-48.283444,-15.737939,GO,Goiás,Centro-Oeste,Não,...,0.00,3.85,71.4,0.0,0.08,25,0.2,0.200000,0.0,2.639057
1,2023,5200258,Águas Lindas de Goiás,Não,-48.283444,-15.737939,GO,Goiás,Centro-Oeste,Não,...,0.00,0.00,59.3,0.0,0.42,25,0.2,0.000000,0.0,3.295837
2,2023,5200258,Águas Lindas de Goiás,Não,-48.283444,-15.737939,GO,Goiás,Centro-Oeste,Não,...,8.33,3.85,87.5,0.0,0.15,25,0.2,0.000000,0.0,3.178054
3,2023,5200258,Águas Lindas de Goiás,Não,-48.283444,-15.737939,GO,Goiás,Centro-Oeste,Não,...,0.00,0.00,77.8,0.0,0.20,25,0.2,0.000000,0.0,2.890372
4,2023,5200258,Águas Lindas de Goiás,Não,-48.283444,-15.737939,GO,Goiás,Centro-Oeste,Não,...,4.17,8.66,54.2,4.8,0.64,20,0.3,0.344828,0.0,3.178054


In [39]:
# =========================================================
# 4) Definir variáveis
# =========================================================
target = "qt_ing"

features_originais = [
    "tp_rede",
    "tp_organizacao_academica",
    "prop_doc_avancado",
    "tp_grau_academico",
    "tp_modalidade_ensino",
    "qt_conc",
    "prop_ing_pp",
    "prop_ing_financiados"
]

# Criar dummies
df_model = pd.get_dummies(df_model, columns=[
    "tp_rede", "tp_organizacao_academica", "tp_grau_academico", "tp_modalidade_ensino"
], drop_first=True)

features_finais = [
    "prop_doc_avancado", "qt_conc", "prop_ing_pp", "prop_ing_financiados"
] + [c for c in df_model.columns if c.startswith(("tp_rede_", "tp_organizacao_academica_", "tp_grau_academico_", "tp_modalidade_ensino_"))]

X = df_model[features_finais]
y = df_model[target]
offset = df_model["offset_log_qtmat"]

# Split
X_train, X_test, y_train, y_test, off_train, off_test = train_test_split(
    X, y, offset, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Colocar de volta em DataFrame para o GLM
X_train_df = pd.DataFrame(X_train_scaled, columns=features_finais, index=X_train.index)
X_test_df = pd.DataFrame(X_test_scaled, columns=features_finais, index=X_test.index)
print("Train shape:", X_train_df.shape)
print("Test shape:", X_test_df.shape)

Train shape: (4341, 12)
Test shape: (1086, 12)


In [41]:
# =========================================================
# 5) Modelo Frequentista (GLM Poisson com offset)
# =========================================================
X_train_glm = sm.add_constant(X_train_df)
X_test_glm = sm.add_constant(X_test_df)

model = sm.GLM(y_train, X_train_glm, 
               family=sm.families.Poisson(), 
               offset=off_train)
result = model.fit()

print(result.summary())

y_pred_glm = result.predict(X_test_glm, offset=off_test)

# Métricas
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

metricas = {
    "glm_poisson": {
        "r2": r2_score(y_test, y_pred_glm),
        "rmse": mean_squared_error(y_test, y_pred_glm),
        "mae": mean_absolute_error(y_test, y_pred_glm)
    }
}

print("Métricas (GLM Poisson):", metricas)


                 Generalized Linear Model Regression Results                  
Dep. Variable:                 qt_ing   No. Observations:                 4341
Model:                            GLM   Df Residuals:                     4328
Model Family:                 Poisson   Df Model:                           12
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -16151.
Date:                sáb, 20 set 2025   Deviance:                       20119.
Time:                        12:55:53   Pearson chi2:                 2.05e+04
No. Iterations:                     6   Pseudo R-squ. (CS):             0.9377
Covariance Type:            nonrobust                                         
                                                                                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

In [43]:
# =========================================================
# 6) Modelo Bayesiano (ADVI com offset)
# =========================================================
with pm.Model() as modelo_bayes:
    X_data = pm.Data("X_data", X_train_scaled)
    y_data = pm.Data("y_data", y_train)
    offset_data = pm.Data("offset_data", off_train)

    beta = pm.Normal("beta", mu=0, sigma=10, shape=X_train.shape[1])
    intercept = pm.Normal("intercept", mu=0, sigma=10)

    mu = pm.math.exp(intercept + pm.math.dot(X_data, beta) + offset_data)

    y_obs = pm.Poisson("y_obs", mu=mu, observed=y_data)

    # ADVI (rápido)
    approx = pm.fit(10000)
    trace = approx.sample(1000)

az.to_netcdf(trace, "modelo_bayesiano_trace.nc")
print(az.summary(trace, hdi_prob=0.95))


Output()

Finished [100%]: Average Loss = 8.2488e+08
arviz - WARNING - Shape validation failed: input_shape: (1, 1000), minimum_shape: (chains=2, draws=4)


            mean     sd  hdi_2.5%  hdi_97.5%  mcse_mean  mcse_sd  ess_bulk  \
beta[0]    0.025  0.529    -0.967      1.081      0.017    0.011     999.0   
beta[1]   -0.406  0.376    -1.128      0.321      0.012    0.009     915.0   
beta[2]   -0.231  0.566    -1.254      0.948      0.018    0.012    1025.0   
beta[3]   -0.226  0.491    -1.109      0.806      0.015    0.010    1020.0   
beta[4]   -0.219  0.464    -1.172      0.605      0.015    0.011    1018.0   
beta[5]    0.766  0.575    -0.344      1.957      0.022    0.013     712.0   
beta[6]   -0.103  0.372    -0.842      0.574      0.013    0.009     827.0   
beta[7]    0.217  0.511    -0.798      1.174      0.015    0.013    1159.0   
beta[8]    0.561  0.511    -0.338      1.631      0.016    0.012    1080.0   
beta[9]   -0.165  0.349    -0.900      0.472      0.010    0.008    1205.0   
beta[10]   0.799  0.525    -0.282      1.718      0.018    0.011     874.0   
beta[11]  -0.911  0.483    -1.851      0.009      0.014    0.011

In [50]:
# =========================================================
# 7) Previsão para o curso alvo (IESB - Ciência de Dados e IA)
# =========================================================
curso_alvo = "Ciência De Dados E Inteligência Artificial"
ies_alvo = "CENTRO UNIVERSITÁRIO DO INSTITUTO DE EDUCAÇÃO SUPERIOR DE BRASÍLIA - IESB"

# Filtrar curso alvo
df_alvo = df[(df["no_curso"].str.contains(curso_alvo, case=False, na=False)) &
             (df["no_ies"].str.contains("IESB", case=False, na=False))]

print("Curso alvo encontrado:")
display(df_alvo[["no_curso", "no_ies", "qt_ing", "qt_mat", "qt_conc"]])

# Preparar dados
df_alvo = preparar_dados(df_alvo)

# Criar dummies
df_alvo = pd.get_dummies(df_alvo, columns=[
    "tp_rede", "tp_organizacao_academica", "tp_grau_academico", "tp_modalidade_ensino"
], drop_first=True)

# Reindexar para bater com treino
X_alvo = df_alvo.reindex(columns=features_finais, fill_value=0)

# Escalar
X_alvo_scaled = scaler.transform(X_alvo)

# Transformar em DataFrame com nomes certos
X_alvo_df = pd.DataFrame(X_alvo_scaled, columns=features_finais, index=X_alvo.index)

# Reindexar para garantir ordem idêntica ao treino
X_alvo_df = X_alvo_df.reindex(columns=X_train_df.columns, fill_value=0)

# Adicionar constante
X_alvo_glm = sm.add_constant(X_alvo_df, has_constant="add")

# Offset do alvo
off_alvo = df_alvo["offset_log_qtmat"]

# ----------------------------
# Previsão GLM (frequentista)
# ----------------------------
prev_glm = result.predict(X_alvo_glm, offset=off_alvo)

# ----------------------------
# Previsão Bayesiana (ADVI)
# ----------------------------
with modelo_bayes:
    pm.set_data({"X_data": X_alvo_scaled, "offset_data": off_alvo})

    # Criar pseudo-trace a partir do approx
    trace_advi = approx.sample(1000)

    # Gerar predições
    pred_samples = pm.sample_posterior_predictive(
        trace_advi, var_names=["y_obs"], random_seed=42, return_inferencedata=True
    )

# Acessar predições corretamente
bayes_samples = pred_samples.posterior_predictive["y_obs"].values.flatten()
bayes_mean = bayes_samples.mean()
bayes_low = np.percentile(bayes_samples, 5)
bayes_high = np.percentile(bayes_samples, 95)

# ----------------------------
# Exibir resultados
# ----------------------------
print("\nPrevisões:")
print("Valor Real 2023:", int(df_alvo["qt_ing"].iloc[0]))
print("GLM Poisson:", float(prev_glm.iloc[0]))
print("Bayesiano (média):", bayes_mean, "IC90%:", bayes_low, "-", bayes_high)


Curso alvo encontrado:


,no_curso,no_ies,qt_ing,qt_mat,qt_conc
208,Ciência De Dados E Inteligência Artificial,CENTRO UNIVERSITÁRIO DO INSTITUTO DE EDUCAÇÃO ...,16,81,18


Sampling: [y_obs]


Output()


Previsões:
Valor Real 2023: 16
GLM Poisson: 46.40257812809445
Bayesiano (média): 12.792942409583045 IC90%: 1.0 - 41.0


In [52]:
# =========================================================
# 8) Salvar Resultados (com coeficientes nomeados)
# =========================================================

# Extrair betas médios do trace bayesiano
betas = trace.posterior["beta"].mean(dim=["chain", "draw"]).values
coef_dict = {var: float(betas[i]) for i, var in enumerate(features_finais)}

# Intercepto
intercept_mean = float(trace.posterior["intercept"].mean().values)
coef_dict["intercept"] = intercept_mean

# Montar dicionário final
resultados = {
    "metricas": metricas,
    "previsoes": {
        "curso_alvo": curso_alvo,
        "ies_alvo": ies_alvo,
        "real_2023": int(df_alvo["qt_ing"].iloc[0]),
        "glm_poisson": float(prev_glm.iloc[0]),
        "bayesian": {
            "media": float(bayes_mean),
            "ic_low": float(bayes_low),
            "ic_high": float(bayes_high),
            "coeficientes": coef_dict
        }
    }
}

# Salvar em JSON
with open("resultados_modelos.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=4, ensure_ascii=False)

print("Resultados salvos em resultados_modelos.json")


Resultados salvos em resultados_modelos.json
